# Exploración de datos
---

**Autor:** Jaime Lozano Cillero

# 0. Preparación del entorno

In [ ]:
from dotenv import load_dotenv
import getpass
import os
from pathlib import Path
from openai import OpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

GENERATION_MODEL = os.getenv("OPENAI_GENERATION_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

client = OpenAI()

RAG_K = 5

PDF_PATH = Path("../../data/raw/TENERIFE.pdf")

print("Modelo generativo:", GENERATION_MODEL)
print("Modelo de embeddings:", EMBEDDING_MODEL)
print("PDF:", PDF_PATH.resolve())

Modelo generativo: gpt-4.1-mini
Modelo de embeddings: text-embedding-3-small
PDF: C:\Users\Jaime\Documents\Pontia\modulo-06-llms\entrega-final\pontia-llms-entrega-final\data\raw\TENERIFE.pdf


In [4]:
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings

llm = init_chat_model(
    GENERATION_MODEL,
    model_provider="openai",
    temperature=0.1,
    max_tokens=1000,
)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# 1. Carga de datos

Se usará el fichero PDF local: `data/raw/TENERIFE.pdf`.

El objetivo es que el contenido acabe convertido en objetos `Document`, que tienen dos partes:

- `page_content`: el texto del documento.
- `metadata`: información adicional, como fuente, página, ruta, fecha, categoría, etc.

In [5]:
from pypdf import PdfReader
from langchain_core.documents import Document

if not PDF_PATH.exists():
    raise FileNotFoundError(f"No se encontró el PDF en {PDF_PATH.resolve()}")

reader = PdfReader(str(PDF_PATH))

docs = [
    Document(
        page_content=page.extract_text().strip(),
        metadata={
            "source": str(PDF_PATH),
            "page": page_number,
            "page_label": reader.page_labels[page_number],
            "total_pages": len(reader.pages),
        },
    )
    for page_number, page in enumerate(reader.pages)
]

print("Número de páginas cargadas:", len(docs))
print("Metadata de la primera página:", docs[0].metadata)
print("Primeros caracteres:\n")
print(docs[0].page_content[:512])

Número de páginas cargadas: 25
Metadata de la primera página: {'source': '..\\..\\data\\raw\\TENERIFE.pdf', 'page': 0, 'page_label': '1', 'total_pages': 25}
Primeros caracteres:

TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo 
dirección Plaza Weyler -


In [6]:
print(docs[7].page_content)

o Playa de Benijo [vídeo - ubicación] 
Playa de arena negra, si el día está despejado, de los atardeceres más 
espectaculares que podéis ver en la isla. 
Para llegar a esta playa, aparcar por aquí y, para acceder a la playa, se accede 
desde aquí (puede ser que os encontréis con el acceso cortado con una valla, 
pero da igual, podéis acceder igualmente). 
• La Orotava 
Este es el pueblo más bonito y maravilloso del mundo (no es porque yo sea de allí).


In [7]:
print(reader.metadata)
print(reader.page_layout)
print(reader.page_mode)

{'/Title': 'Microsoft Word - TENERIFE.docx', '/Creator': 'Microsoft Word', '/CreationDate': "D:20250713200001+00'00'", '/ModDate': "D:20250713200001+00'00'"}
None
None


# 2. Chunking

La guía tiene poco texto así que usaremos chunks de pequeño tamaño.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    add_start_index=True,
)

splits = text_splitter.split_documents(docs)

# Añadimos un identificador simple a cada chunk para poder citarlo luego.
for i, doc in enumerate(splits):
    doc.metadata["chunk_id"] = i
    doc.metadata["source_name"] = PDF_PATH.name

print("Número de chunks:", len(splits))
print("\nEjemplo de chunk:\n")
print(splits[0].page_content)
print("\nMetadata:", splits[0].metadata)

Número de chunks: 47

Ejemplo de chunk:

TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo

Metadata: {'source': '..\\..\\data\\raw\\TENERIFE.pdf', 'page': 0, 'page_label': '1', 'total_pages': 25, 'start_index': 0, 'chunk_id': 0, 'source_name': 'TENERIFE.pdf'}


In [9]:
for doc in splits:
    print(f"Chunk {doc.metadata['chunk_id']} | Página {doc.metadata.get('page')} | Caracteres: {len(doc.page_content)}")

Chunk 0 | Página 0 | Caracteres: 486
Chunk 1 | Página 0 | Caracteres: 472
Chunk 2 | Página 1 | Caracteres: 86
Chunk 3 | Página 2 | Caracteres: 439
Chunk 4 | Página 2 | Caracteres: 158
Chunk 5 | Página 3 | Caracteres: 379
Chunk 6 | Página 4 | Caracteres: 422
Chunk 7 | Página 4 | Caracteres: 431
Chunk 8 | Página 5 | Caracteres: 124
Chunk 9 | Página 7 | Caracteres: 455
Chunk 10 | Página 8 | Caracteres: 490
Chunk 11 | Página 8 | Caracteres: 495
Chunk 12 | Página 8 | Caracteres: 253
Chunk 13 | Página 9 | Caracteres: 219
Chunk 14 | Página 10 | Caracteres: 486
Chunk 15 | Página 10 | Caracteres: 430
Chunk 16 | Página 10 | Caracteres: 394
Chunk 17 | Página 11 | Caracteres: 488
Chunk 18 | Página 11 | Caracteres: 469
Chunk 19 | Página 11 | Caracteres: 458
Chunk 20 | Página 11 | Caracteres: 443
Chunk 21 | Página 12 | Caracteres: 162
Chunk 22 | Página 13 | Caracteres: 373
Chunk 23 | Página 14 | Caracteres: 474
Chunk 24 | Página 14 | Caracteres: 493
Chunk 25 | Página 14 | Caracteres: 22
Chunk 26 | P

# 3. Embeddings y vector store

El objetivo del RAG es recuperar información relacionada con las consultas variables del usuario. Para ello, necesitamos comparar el signficado de la pregunta con el significado de nuestros textos (chunks) para recuperar los que sean más parecidos. Esos chunks recuperados serán provistos al LLM como contexto de tal manera que lo integre en su respuesta al usuario y dé como resultado una respuesta relevante.

Para poder comparar significados, cada texto será transformado a embedding y se compararán usando una medida de similaridad.

Se usará la clase `InMemoryVectorStore` para la transformación y almacenamiento de los embeddings en memoria.

In [10]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
document_ids = vector_store.add_documents(splits)

print("Documentos indexados:", len(document_ids))
print("Primeros IDs:", document_ids[:3])

Documentos indexados: 47
Primeros IDs: ['4ebcaf7f-2a8a-4bad-b73d-c44de7191483', 'b74f91bd-30b1-424c-96f7-4725a61abe91', '9439220b-2c83-4053-99f1-f37fe18197ff']


# 4. Recuperación 

Comprobar el mecanismo de recuperación para verificar que se recuperan chunks relevantes con la consulta del usuario.

In [11]:
def mostrar_documentos_recuperados(documentos):
    for i, doc in enumerate(documentos, start=1):
        print(f"Resultado {i}")
        print("Fuente:", doc.metadata.get("source_name"))
        print("Página:", doc.metadata.get("page"))
        print("Chunk:", doc.metadata.get("chunk_id"))
        print(doc.page_content)
        print("-" * 80)

pregunta = "¿Dònde puedo comer sushi en Tenerife?"
retrieved_docs = vector_store.similarity_search(pregunta, k=3)

mostrar_documentos_recuperados(retrieved_docs)

Resultado 1
Fuente: TENERIFE.pdf
Página: 24
Chunk: 45
▪ Arepera La Carajita (entre La Orotava y el Puerto de la Cruz 
[ubicación]. Este sitio tiene las mejores arepas que podréis comer en 
vuestra vida y son baratísimas. Recomendación: La Fogalera y Reina 
Pepiada. Y pedir de entrantes una degustación de cachapitas). 
Importante ir con hambre.  
▪ Atuvera (entre La Orotava y el Puerto de la Cruz [ubicación]). 
Recomendable reservar. 
▪ Restaurante Fujiyama (sushi en la zona de Santa Cruz [ubicación])
--------------------------------------------------------------------------------
Resultado 2
Fuente: TENERIFE.pdf
Página: 23
Chunk: 44
El Paraíso (espectacular la salsa de aguacate de aquí. 
Recomendación: menú feliz de bocadillo de pollo + Nestea de 
mango-piña) - Cafetería Zumería Mango - El Track 
 
 
Nota:   los bocadillos de pollo del norte de Tenerife no son como los bocadillos 
que se hacen en otros lugares de España.   Recomiendo que lo prueben, está 
increíble.
-------------------

El primer chunk recuperado sí incluye la respuesta. El resto son ruido.

Sería conveniente crear chunks con otra estrategia. Una estrategia que tuviera en cuenta bloques de información mejor delimitados, y que no dejen las ideas a medias.

# 5. Añadir los chunks al contexto del LLM

Se añaden los chunks recuperados al prompt que se envía al LLM.

Queremos que el modelo:

1. Responda usando solo el contexto.
2. Diga que no tiene información si el contexto no contiene la respuesta.
3. Cite las fuentes utilizadas.

In [19]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Eres un guía turístico con más de 20 años de experiencia en la isla Tenerife."
            "Responde en español utilizando expresiones típicas canarias y usando únicamente el contexto proporcionado."
            "Si el contexto no contiene la respuesta, di claramente: 'No tengo información suficiente en la base documental'. "
            "No inventes datos. Cita las fuentes usadas al final.",
        ),
        (
            "user",
            "Pregunta:\n{question}\n\nContexto recuperado:\n{context}\n\nRespuesta:",
        ),
    ]
)

In [13]:
def format_docs_with_sources(documentos):
    bloques = []
    for i, doc in enumerate(documentos, start=1):
        source = doc.metadata.get("source_name", "fuente desconocida")
        page = doc.metadata.get("page", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        bloques.append(
            f"[Fuente {i}: {source}, página {page}, chunk {chunk_id}]\n{doc.page_content}"
        )
    return "\n\n".join(bloques)


def answer_with_rag(question: str, k: int = 3, show_context: bool = False):
    retrieved_docs = vector_store.similarity_search(question, k=k)
    context = format_docs_with_sources(retrieved_docs)

    if show_context:
        print("CONTEXTO RECUPERADO")
        print(context)
        print("=" * 80)

    messages = rag_prompt.invoke({"question": question, "context": context})
    answer = llm.invoke(messages)

    return {
        "question": question,
        "answer": answer.content,
        "context": retrieved_docs,
        "prompt": messages,
    }

In [20]:
result = answer_with_rag(
    pregunta,
    k=RAG_K,
    show_context=True,
)

print(result["answer"])

CONTEXTO RECUPERADO
[Fuente 1: TENERIFE.pdf, página 24, chunk 45]
▪ Arepera La Carajita (entre La Orotava y el Puerto de la Cruz 
[ubicación]. Este sitio tiene las mejores arepas que podréis comer en 
vuestra vida y son baratísimas. Recomendación: La Fogalera y Reina 
Pepiada. Y pedir de entrantes una degustación de cachapitas). 
Importante ir con hambre.  
▪ Atuvera (entre La Orotava y el Puerto de la Cruz [ubicación]). 
Recomendable reservar. 
▪ Restaurante Fujiyama (sushi en la zona de Santa Cruz [ubicación])

[Fuente 2: TENERIFE.pdf, página 23, chunk 44]
El Paraíso (espectacular la salsa de aguacate de aquí. 
Recomendación: menú feliz de bocadillo de pollo + Nestea de 
mango-piña) - Cafetería Zumería Mango - El Track 
 
 
Nota:   los bocadillos de pollo del norte de Tenerife no son como los bocadillos 
que se hacen en otros lugares de España.   Recomiendo que lo prueben, está 
increíble.

[Fuente 3: TENERIFE.pdf, página 10, chunk 16]
Charco (los helados del Freddino no están mal po

Probamos a preguntar algo un poco difuso:

In [21]:
result = answer_with_rag(
    "Me gustaría bañarme en el mar",
    k=RAG_K,
    show_context=True,
)

print(result["answer"])

CONTEXTO RECUPERADO
[Fuente 1: TENERIFE.pdf, página 19, chunk 32]
o Playa del Camisón 
Esta es una playa pequeñita que a nosotros en concreto nos gusta mucho.

[Fuente 2: TENERIFE.pdf, página 9, chunk 13]
o La Playa de Los Patos 
o Playa del Ancón 
 
Nota: Para llegar estas playas hay que caminar un rato y algunas son de difícil acceso 
(la de Los Patos). También hay que ir a estas playas cuando la marea esté baja, si no

[Fuente 3: TENERIFE.pdf, página 11, chunk 20]
San Miguel, donde se encuentran las Piscinas Naturales del Caletón y donde 
si queréis os podéis pegar un chuso (un baño, para que os vayáis empapando 
de jerga canaria). Luego iría dirección hacia la Plaza Juan González de la Torre 
y subiría hacia la Plaza de la Libertad, cerveza bien fresquita y callejear un 
poco por ahí. En Garachico se encuentra una pizzería que es buenísima, su 
nombre es Pizzería Rugantino (consultad los horarios).

[Fuente 4: TENERIFE.pdf, página 10, chunk 15]
Santo Domingo, la calle de San Telmo 

Probamos a preguntar algo que no incluye la fuente documental.

In [16]:
result = answer_with_rag(
    "Me gustaría ir a un circuito de Formula 1",
    k=RAG_K,
)

print(result["answer"])

No tengo información suficiente en la base documental sobre la existencia de un circuito de Fórmula 1 en Tenerife o en las Islas Canarias. 

Fuentes consultadas: TENERIFE.pdf, páginas 8, 10, 13, 14, 16.
